# Recalibrating Sirius (Sector 6) with `mundey` v2.0

This notebook demonstrates the functional `mundey` v2.0 calibration workflow on the bundled TESS Sector 6 Target Pixel File (TPF) and collateral fixtures for Sirius (`TIC 322899250`), followed by background estimation with `tessbkgd` and halo photometry with `halophot`.

In [ ]:
import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np

import mundey

In [ ]:
tpffile = "tess2018349182459-s0006-0000000322899250-0126-s_tp.fits"
raw_tpf = lk.read(tpffile)
raw_tpf.plot(aspect=0.02, scale="log", interpolation="nearest", title="Sirius - before");

## Use `mundey.recalibrate_tpf` to recalibrate the TPF

`mundey.recalibrate_tpf` pairs science and collateral cadences by exact integer `CADENCENO` (`cadence_match="cadenceno"`), fits the orthonormal Legendre 1D black in double precision, inverts LDE undershoot and non-linearity in physical readout order, and automatically selects concavity-guarded alternate smear repair (`smear="automatic"`) because the Sirius stamp reaches CCD row 2048.

Note that the bundled 100-row Sector 6 TPF (`QUALITY == 0` filtered) spans `CADENCENO = 173388..173494` with a 7-cadence mid-orbit gap (`173464..173470`), whereas the bundled 100-row `_col.fits` tables span `173387..173486`. Exact `CADENCENO` pairing aligns the 92 overlapping cadences cleanly and masks the 8 unmatched trailing cadences (`173487..173494`, recorded in `CADMISS = 8`) to `NaN`.

In [ ]:
cal_tpf = mundey.recalibrate_tpf(
    tpffile,
    ddir=".",
    smear="automatic",
    cadence_match="cadenceno",
    missing_cadence="nan",
    offline=True,
    verbose=True,
)

In [ ]:
smear_err = mundey.read_smear_err_column(cal_tpf, quality_masked=False)
pixel_quality = mundey.read_pixel_quality(cal_tpf)

print(f"HALOCVER = {cal_tpf.meta['HALOCVER']}, BKGSUB = {cal_tpf.meta['BKGSUB']}")
print(
    f"CADMATCH = {cal_tpf.hdu[0].header['CADMATCH']}, "
    f"CADMISS = {cal_tpf.hdu[0].header['CADMISS']}"
)
print(f"SMEAR_ERR shape = {smear_err.shape}, PIXEL_QUALITY shape = {pixel_quality.shape}")
print(f"Total calibrated flux sum = {np.nansum(cal_tpf.flux.value):.3e} e-/s")

In [ ]:
cal_tpf.plot(aspect=0.02, scale="log", interpolation="nearest", title="Sirius - after");

## Optional downstream workflow: `tessbkgd` and `halophot`

`mundey` v2.0 returns un-background-subtracted calibrated flux (`FLUX_BKG == 0`, `BKGSUB == False`, `BACKAPP == False`) so `tessbkgd` can estimate and subtract the scattered-light background directly without double-counting SPOC's background polynomial.

In [ ]:
# Uncomment when tessbkgd and halophot are installed in the active environment:
# import tessbkgd
# import halophot
#
# valid_cads = np.all(np.isfinite(cal_tpf.flux.value), axis=(1, 2))
# sub_tpf = cal_tpf[valid_cads]
# sub_tpf.__class__ = tessbkgd.tessbkgd.bkgd_tpf
# newbkgd, bkgdmask = sub_tpf.get_bkgd(verbose=True)
# sub_tpf.flux = sub_tpf.flux - newbkgd
# sub_tpf.flux_bkg = newbkgd
#
# sub_tpf.__class__ = halophot.halo_tools.halo_tpf_tess
# halomask = np.zeros(sub_tpf.flux.shape[1:], dtype=bool)
# y0 = sub_tpf.hdu[2].header["CRVAL2P"] - 1
# halomask[(1700 - y0) : (1875 - y0), :] = True
# sub_tpf.plot(
#     aspect=0.02,
#     scale="log",
#     interpolation="nearest",
#     title="Sirius - halo aperture",
#     aperture_mask=halomask,
# );